In [ ]:
import cv2
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
sns.set_style('whitegrid')
from sklearn.metrics import confusion_matrix , classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , Flatten , Conv2D , MaxPooling2D , Dropout , Activation , BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam , Adamax
from tensorflow.keras import regularizers

#Warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import tensorflow as tf
#tf.keras.mixed_precision.set_global_policy('mixed_float16')

import tensorflow as tf
import numpy as np
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, ReLU, Add
from tensorflow.keras.models import Model
from tensorflow.keras.losses import KLDivergence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import DenseNet121, ResNet50V2
from tensorflow.keras.layers import GlobalAveragePooling2D
import copy
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, ReLU, Add
from tensorflow.keras.models import Model
from tensorflow.keras.losses import KLDivergence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import DenseNet169, MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D
import copy

import tensorflow as tf
from tensorflow.keras import layers
import tensorflow as tf
from tensorflow.keras import layers


In [ ]:
X_train_s = np.load('X.npy')
y_train_s = np.load('Y.npy')

X_train_s.shape, y_train_s.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_train_s, y_train_s, test_size=0.2, random_state=42)
X_train_s, X_val_s, y_train_s, y_val_s = train_test_split(X_train_s, y_train_s, test_size=0.2, random_state=42)

X_train_s.shape,X_test_s.shape, y_train_s.shape,y_test_s.shape, y_val_s.shape,y_val_s.shape

In [ ]:
import tensorflow as tf, numpy as np
tf.keras.backend.clear_session()
tf.random.set_seed(0); np.random.seed(0)

# ---------- small utils ----------
def _l2norm_per_sample(x, eps=1e-12):
    # x: [N,H,W,C]
    denom = tf.maximum(tf.sqrt(tf.reduce_sum(tf.square(x), axis=[1,2,3], keepdims=True)), eps)
    return x / denom

def top2_margin(logits):
    v = tf.nn.top_k(logits, k=2).values
    return v[:,0] - v[:,1]

def certified_radius_L2(logits, L=1.0):
    return (top2_margin(logits) / (2.0 * L)).numpy()

# ---------- GroupSort ----------
'''class GroupSort(tf.keras.layers.Layer):
    def __init__(self, group_size=2, **kw):
        super().__init__(**kw); self.g = group_size
    def call(self, x):
        c = tf.shape(x)[-1]
        tf.debugging.assert_equal(c % self.g, 0, "channels % group_size must be 0")
        shp = tf.concat([tf.shape(x)[:-1], [c//self.g, self.g]], 0)
        xs = tf.sort(tf.reshape(x, shp), axis=-1)
        return tf.reshape(xs, tf.shape(x))'''

class GroupSort(tf.keras.layers.Layer):
    def __init__(self, group_size=2, **kw):
        super().__init__(**kw)
        self.g = int(group_size)

    def build(self, input_shape):
        c = input_shape[-1]
        # If channel count is known at build time, check divisibility once
        if c is not None and c % self.g != 0:
            raise ValueError(f"GroupSort: channels ({c}) must be divisible by group_size ({self.g}).")
        super().build(input_shape)

    def call(self, x):
        c = tf.shape(x)[-1]
        tf.debugging.assert_equal(
            c % self.g, 0,
            message="GroupSort: channels % group_size must be 0"
        )
        new_shape = tf.concat([tf.shape(x)[:-1], [c // self.g, self.g]], axis=0)
        xg = tf.reshape(x, new_shape)          # [..., groups, g]
        xs = tf.sort(xg, axis=-1)              # sort within each group (1-Lipschitz)
        return tf.reshape(xs, tf.shape(x))     # back to original shape

    def compute_output_shape(self, input_shape):
        # same spatial/chan shape as input
        return tf.TensorShape(input_shape)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"group_size": self.g})
        return cfg


# ---------- Orthonormal 1x1 (Constant initializer, no .numpy()) ----------
'''class OrthoMix1x1(tf.keras.layers.Layer):
    def build(self, input_shape):
        c = int(input_shape[-1])
        q, _ = tf.linalg.qr(tf.random.normal([c, c]))
        k = tf.reshape(q, [1,1,c,c])
        self.W = self.add_weight(
            name="W",
            shape=[1,1,c,c],
            initializer=tf.keras.initializers.Constant(tf.cast(k, tf.float32)),
            trainable=False,
        )
    def call(self, x): return tf.nn.conv2d(x, self.W, strides=1, padding="SAME")'''


#################################

import tensorflow as tf
from tensorflow.keras.layers import Layer, DepthwiseConv2D, Conv2D, GlobalAveragePooling2D, Dense

class TrainableSpatialAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableSpatialAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same', 
                                  trainable=True)
        super(TrainableSpatialAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.convolution(inputs)
        return tf.multiply(inputs, attention_weights)

class TrainableChannelAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableChannelAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.global_avg_pooling = GlobalAveragePooling2D()
        self.dense1 = Dense(units=input_shape[-1] // 8, activation='relu', trainable=True)
        self.dense2 = Dense(units=input_shape[-1], activation='sigmoid', trainable=True)
        super(TrainableChannelAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pooling(inputs)
        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)
        return tf.multiply(inputs, channel_attention_weights)

class TrainableCombinedAttentionLayer(Layer):
    def __init__(self, spatial_noise_factor=0.6, channel_noise_factor=0.6, num_layers=3, **kwargs):
        super(TrainableCombinedAttentionLayer, self).__init__(**kwargs)
        self.spatial_attentions = [TrainableSpatialAttentionLayer() for _ in range(num_layers)]
        self.channel_attentions = [TrainableChannelAttentionLayer() for _ in range(num_layers)]
        
        # Trainable weights for noise factors
        self.spatial_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='spatial_noise_weight')
        self.channel_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='channel_noise_weight')
        
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor

    def call(self, inputs):
        spatial_attention_output = inputs
        channel_attention_output = inputs

        for spatial_attention, channel_attention in zip(self.spatial_attentions, self.channel_attentions):
            # Spatial Attention
            spatial_attention_output = spatial_attention(spatial_attention_output)

            # Channel Attention
            channel_attention_output = channel_attention(channel_attention_output)

        # Generate spatial and channel noise
        spatial_noise = tf.random.normal(shape=tf.shape(spatial_attention_output), mean=0, 
                                         stddev=self.spatial_noise_factor)
        channel_noise = tf.random.normal(shape=tf.shape(channel_attention_output), mean=0, 
                                         stddev=self.channel_noise_factor)

        # Scale the noise with trainable weights
        spatial_noise *= self.spatial_noise_weight + (spatial_noise * self.spatial_noise_weight)
        channel_noise *= self.channel_noise_weight + (channel_noise * self.channel_noise_weight)

        # Add spatial attention noise
        spatial_attention_output += spatial_noise

        # Add channel attention noise
        channel_attention_output += channel_noise

        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention1 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention2 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention3 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention4 = tf.multiply(combined_attention1, combined_attention2)
        combined_attention = tf.multiply(combined_attention3, combined_attention4)
        
        return tf.multiply(inputs, combined_attention)



##################################

# --- SAFE OrthoMix1x1: NumPy QR in build, Constant initializer with np array
class OrthoMix1x1(tf.keras.layers.Layer):
    """Frozen orthonormal 1x1 mix (energy-preserving, 1-Lipschitz)."""
    def build(self, input_shape):
        self.att2  = TrainableCombinedAttentionLayer()
        c = int(input_shape[-1])
        # NumPy QR so no TF graph tensors are captured
        q, _ = np.linalg.qr(np.random.randn(c, c))
        k = q.astype(np.float32).reshape(1, 1, c, c)
        self.W = self.add_weight(
            name="W",
            shape=[1, 1, c, c],
            initializer=tf.keras.initializers.Constant(k),
            trainable=False,
        )
    def call(self, x):
        a = tf.nn.conv2d(x, self.W, strides=1, padding="SAME")
        a = a + self.att2(a, training=True)   # attention after conv2
        return a



# ---------- Spatial-RP (frozen) ----------
def _make_block_orthogonal_kernel(kh, kw, in_ch, out_ch, block=512, seed=123):
    K = kh*kw*in_ch
    rng = tf.random.Generator.from_seed(seed)
    parts, start = [], 0
    while start < K:
        b = min(block, K-start)
        M = rng.normal([b, out_ch])
        if b < out_ch:
            pad = out_ch - b
            M = tf.concat([M, rng.normal([pad, out_ch])], axis=0)
        q, _ = tf.linalg.qr(M)
        parts.append(q[:b,:])
        start += b
    return tf.reshape(tf.concat(parts, axis=0), [kh,kw,in_ch,out_ch])

'''class FrozenRPCONV(tf.keras.layers.Layer):
    def __init__(self, filters, kernel_size=3, strides=1, padding="SAME", **kw):
        super().__init__(**kw)
        self.f = int(filters); self.ks = kernel_size if isinstance(kernel_size,(tuple,list)) else (kernel_size,kernel_size)
        self.s = int(strides); self.p = padding
    def build(self, input_shape):
        in_ch = int(input_shape[-1]); kh, kw = self.ks
        W0 = _make_block_orthogonal_kernel(kh, kw, in_ch, self.f)
        self.W0 = self.add_weight(
            "W0",
            shape=W0.shape,
            initializer=tf.keras.initializers.Constant(tf.cast(W0, tf.float32)),
            trainable=False,
        )
        self.H = int(input_shape[1]); self.W = int(input_shape[2])
    def call(self, x, training=None):
        OH = (self.H + self.s - 1)//self.s; OW = (self.W + self.s - 1)//self.s
        u = tf.random.normal([1, OH, OW, self.f])
        for _ in range(2):
            v = _l2norm_vec(tf.nn.conv2d_transpose(u, self.W0, output_shape=tf.shape(x),
                                                   strides=[1,self.s,self.s,1], padding=self.p))
            u = _l2norm_vec(tf.nn.conv2d(v, self.W0, strides=[1,self.s,self.s,1], padding=self.p))
        sigma = tf.reduce_sum(u * tf.nn.conv2d(v, self.W0, strides=[1,self.s,self.s,1], padding=self.p)) + 1e-6
        W_sn = self.W0 / tf.maximum(sigma, 1.0)
        return tf.nn.conv2d(x, W_sn, strides=[1,self.s,self.s,1], padding=self.p)'''

# --- SAFE FrozenRPCONV: handles out_ch > K ---
class FrozenRPCONV(tf.keras.layers.Layer):
    """
    Frozen random-projection conv (≤1-Lip after per-call scaling).
    If out_ch <= K=kh*kw*Cin: use orthonormal columns via QR.
    Else: use column-normalized random columns (still OK; we SN-scale each call).
    """
    def __init__(self, filters, kernel_size=3, strides=1, padding="SAME", **kw):
        super().__init__(**kw)
        self.f = int(filters)
        self.ks = kernel_size if isinstance(kernel_size, (tuple, list)) else (kernel_size, kernel_size)
        self.s = int(strides)
        self.p = padding
        self.att2  = TrainableCombinedAttentionLayer()

    def build(self, input_shape):
        kh, kw = self.ks
        Cin = int(input_shape[-1])
        K = kh * kw * Cin
        f = self.f

        if f <= K:
            # Orthogonal columns in R^K
            M = np.random.randn(K, f)
            q, _ = np.linalg.qr(M)              # [K, K]
            W0_mat = q[:, :f]                   # [K, f]
        else:
            # Cannot have >K orthonormal columns in R^K → use column-normalized random
            M = np.random.randn(K, f)
            M /= (np.linalg.norm(M, axis=0, keepdims=True) + 1e-8)
            W0_mat = M                          # [K, f]

        W0 = W0_mat.astype(np.float32).reshape(kh, kw, Cin, f)
        self.W0 = self.add_weight(
            name = "W0",
            shape=W0.shape,
            initializer=tf.keras.initializers.Constant(W0),
            trainable=False,
        )
        self.H = int(input_shape[1]); self.W = int(input_shape[2])

    '''def call(self, x, training=None):
        OH = (self.H + self.s - 1)//self.s
        OW = (self.W + self.s - 1)//self.s
        u = tf.random.normal([1, OH, OW, self.f])
        # 1–2 power-iter steps → safe upper bound on conv operator norm
        for _ in range(2):
            v = _l2norm_vec(tf.nn.conv2d_transpose(u, self.W0, output_shape=tf.shape(x),
                                                   strides=[1, self.s, self.s, 1], padding=self.p))
            u = _l2norm_vec(tf.nn.conv2d(v, self.W0, strides=[1, self.s, self.s, 1], padding=self.p))
        sigma = tf.reduce_sum(u * tf.nn.conv2d(v, self.W0, strides=[1, self.s, self.s, 1], padding=self.p)) + 1e-6
        W_sn = self.W0 / tf.maximum(sigma, 1.0)
        return tf.nn.conv2d(x, W_sn, strides=[1, self.s, self.s, 1], padding=self.p)'''
    def call(self, x, training=None):
        N  = tf.shape(x)[0]
        OH = (self.H + self.s - 1)//self.s
        OW = (self.W + self.s - 1)//self.s
    
        u = tf.random.normal([N, OH, OW, self.f])  # match batch size
        for _ in range(2):
            v = _l2norm_per_sample(
                tf.nn.conv2d_transpose(
                    u, self.W0, output_shape=tf.shape(x),
                    strides=[1, self.s, self.s, 1], padding=self.p
                )
            )
            u = _l2norm_per_sample(
                tf.nn.conv2d(v, self.W0, strides=[1, self.s, self.s, 1], padding=self.p)
            )
    
        Wv   = tf.nn.conv2d(v, self.W0, strides=[1, self.s, self.s, 1], padding=self.p)
        sigma = tf.reduce_mean(tf.reduce_sum(u * Wv, axis=[1,2,3])) + 1e-6
        W_sn  = self.W0 / tf.maximum(sigma, 1.0)
        a = tf.nn.conv2d(x, W_sn, strides=[1, self.s, self.s, 1], padding=self.p)
        a = a + self.att2(a, training=True)   # attention after conv2
        return a



# ---------- ECONV (no add_weight in call) ----------
def _pad_kernel_to(H, W, k):
    kh, kw = k.shape[0], k.shape[1]
    return tf.pad(k, [[0, H-kh],[0, W-kw],[0,0],[0,0]])

def _fft2_kernel(kpad):            # -> [H,W,Cout,Cin] complex
    K = tf.signal.fft2d(tf.cast(kpad, tf.complex64))
    return tf.transpose(K, [0,1,3,2])

def _max_sigma_exact(K_hw_co_ci):
    H = tf.shape(K_hw_co_ci)[0]; W = tf.shape(K_hw_co_ci)[1]
    M = tf.reshape(K_hw_co_ci, [H*W, tf.shape(K_hw_co_ci)[2], tf.shape(K_hw_co_ci)[3]])
    s = tf.linalg.svd(M, compute_uv=False)
    return tf.reduce_max(tf.reduce_max(s, axis=-1))

class ECONV2D(tf.keras.layers.Layer):
    def __init__(self, filters, kernel_size=3, strides=1, use_bias=True, mode="pi", **kw):
        super().__init__(**kw)
        self.f=int(filters); self.ks=kernel_size if isinstance(kernel_size,(tuple,list)) else (kernel_size,kernel_size)
        self.s=int(strides); self.use_bias=use_bias; self.mode=mode
        self.u=None
        self.att2  = TrainableCombinedAttentionLayer()
    def build(self, input_shape):
        c=int(input_shape[-1])
        self.W = self.add_weight(name="W", shape=[self.ks[0], self.ks[1], c, self.f],
                                 initializer=tf.keras.initializers.HeNormal(), trainable=True)
        self.b = self.add_weight(name="b", shape=[self.f], initializer="zeros", trainable=True) if self.use_bias else None
        self.H = int(input_shape[1]); self.Ws = int(input_shape[2])
        # Pre-create PI vector to avoid add_weight in call()
        OH = (self.H + self.s - 1)//self.s; OW = (self.Ws + self.s - 1)//self.s
        self.u = self.add_weight(name="u", shape=[1, OH, OW, self.f],
                                 initializer=tf.initializers.RandomNormal(stddev=1.0),
                                 trainable=False)
    def _scale_exact(self, x):
        kpad = _pad_kernel_to(self.H, self.Ws, self.W)
        Kf = _fft2_kernel(kpad)
        sigma = tf.cast(_max_sigma_exact(Kf), tf.float32)
        sigma = tf.maximum(sigma, 1.0)
        return self.W / sigma
    '''def _scale_pi(self, x):
        u = self.u
        for _ in range(3):
            v = _l2norm_vec(tf.nn.conv2d_transpose(u, self.W, output_shape=tf.shape(x),
                                                   strides=[1,self.s,self.s,1], padding="SAME"))
            u = _l2norm_vec(tf.nn.conv2d(v, self.W, strides=[1,self.s,self.s,1], padding="SAME"))
        self.u.assign(u)
        sigma = tf.reduce_sum(u * tf.nn.conv2d(v, self.W, strides=[1,self.s,self.s,1], padding="SAME")) + 1e-5
        return self.W / tf.maximum(sigma, 1.0)'''

    def _scale_pi(self, x, iters=5):
        # self.u was created in build(): shape [1, OH, OW, C_out]
        N = tf.shape(x)[0]
        u = tf.tile(self.u, [N, 1, 1, 1])  # match batch
    
        for _ in range(iters):
            v = _l2norm_per_sample(
                tf.nn.conv2d_transpose(
                    u, self.W, output_shape=tf.shape(x),
                    strides=[1, self.s, self.s, 1], padding="SAME"
                )
            )
            u = _l2norm_per_sample(
                tf.nn.conv2d(v, self.W, strides=[1, self.s, self.s, 1], padding="SAME")
            )
    
        # Update cached direction with batch-mean (keeps shape [1,OH,OW,C_out])
        self.u.assign(tf.reduce_mean(u, axis=0, keepdims=True))
    
        Wv   = tf.nn.conv2d(v, self.W, strides=[1, self.s, self.s, 1], padding="SAME")

        #Wv = Wv + self.att2(Wv)   # attention after conv2
        # Batch-mean Rayleigh quotient (scalar), add small slack
        sigma = tf.reduce_mean(tf.reduce_sum(u * Wv, axis=[1,2,3])) + 1e-5
        return self.W / tf.maximum(sigma, 1.0)

    def call(self, x, training=None):
        W_scaled = self._scale_exact(x) if self.mode=="exact" else self._scale_pi(x)
        z = tf.nn.conv2d(x, W_scaled, strides=[1,self.s,self.s,1], padding="SAME")
        z = z + self.att2(z, training=True)   # attention after conv2
        if self.b is not None: z = tf.nn.bias_add(z, self.b)
        return z

# ---------- safe 2D DCT/IDCT ----------
def dct2_safe(x):
    t = tf.transpose(x, [0,2,3,1]); t = tf.signal.dct(t, type=2, norm='ortho', axis=-1)
    t = tf.transpose(t, [0,3,1,2]); t = tf.transpose(t, [0,1,3,2])
    t = tf.signal.dct(t, type=2, norm='ortho', axis=-1)
    return tf.transpose(t, [0,1,3,2])

def idct2_safe(X):
    t = tf.transpose(X, [0,1,3,2]); t = tf.signal.idct(t, type=2, norm='ortho', axis=-1)
    t = tf.transpose(t, [0,1,3,2]); t = tf.transpose(t, [0,2,3,1])
    t = tf.signal.idct(t, type=2, norm='ortho', axis=-1)
    return tf.transpose(t, [0,3,1,2])

# ---------- FreqRPConv ----------
class FreqRPConv(tf.keras.layers.Layer):
    def __init__(self, filters, stride=1, keep_ratio=0.4, seed=123, econv_mode="pi", **kw):
        super().__init__(**kw)
        self.f=int(filters); self.s=int(stride)
        self.keep_ratio=keep_ratio; self.seed=seed; self.econv_mode=econv_mode
        self.mask=None; self.mix=OrthoMix1x1(); self.post=None
        self.att2  = TrainableCombinedAttentionLayer()
    def build(self, input_shape):
        H, W = int(input_shape[1]), int(input_shape[2])
        rng = np.random.default_rng(self.seed)
        M = np.zeros((H, W), dtype=np.float32)
        cy, cx = H//2, W//2
        ry = max(1, int(self.keep_ratio * cy)); rx = max(1, int(self.keep_ratio * cx))
        Y, X = np.ogrid[:H, :W]; M[(((Y-cy)/ry)**2 + ((X-cx)/rx)**2) <= 1.0] = 1.0
        k = max(1, int(0.05*H))
        for _ in range(3):
            y0 = rng.integers(0, H-k); M[y0:y0+k, :] = np.maximum(M[y0:y0+k, :], 0.7)
        self.mask = self.add_weight(name="M", shape=[H,W], initializer=tf.keras.initializers.Constant(M), trainable=False)
        self.post = ECONV2D(self.f, kernel_size=1, strides=self.s, use_bias=True, mode=self.econv_mode)
    def call(self, x, training=None):
        X = dct2_safe(tf.cast(x, tf.float32))
        X = X * tf.reshape(self.mask, [1, tf.shape(x)[1], tf.shape(x)[2], 1])
        x_rec = idct2_safe(X)
        x_mix = self.mix(x_rec)
        a = self.post(x_mix)
        a = a + self.att2(a, training=True)   # attention after conv2
        return a

# ---------- Tri-branch convex gate ----------
class TriBranchGate(tf.keras.layers.Layer):
    def __init__(self, C, **kw):
        super().__init__(**kw); self.C=C; self.logits=None
    def build(self, _):
        self.logits = self.add_weight(name="gate_logits", shape=[3, self.C],
                                      initializer=tf.zeros_initializer(), trainable=True)
    def call(self, z_f, z_s, z_l):
        a = tf.nn.softmax(self.logits, axis=0)            # [3,C]
        a = tf.reshape(a, [3,1,1,1,self.C])
        return a[0]*z_f + a[1]*z_s + a[2]*z_l

# ---------- Combined first conv (tri-branch) ----------
class CombinedConv(tf.keras.layers.Layer):
    def __init__(self, filters, stride=1, econv_mode="pi", enable_freq=True, enable_spatial=True, **kw):
        super().__init__(**kw)
        self.f=int(filters); self.stride=int(stride)
        self.enable_freq=enable_freq; self.enable_spatial=enable_spatial
        self.freq = FreqRPConv(filters, stride=stride, econv_mode=econv_mode) if enable_freq else None
        self.mix  = OrthoMix1x1() if enable_spatial else None
        self.srp  = FrozenRPCONV(filters, kernel_size=3, strides=stride) if enable_spatial else None
        self.learn= ECONV2D(filters, kernel_size=3, strides=stride, use_bias=True, mode=econv_mode)
        self.gate = TriBranchGate(self.f)
        self.att2  = TrainableCombinedAttentionLayer()
    def build(self, input_shape):
        # Pre-build sublayers to avoid any lazy variable creation in call()
        self.learn.build(input_shape)
        if self.enable_freq:
            self.freq.build(input_shape)
        if self.enable_spatial:
            self.mix.build(input_shape)
            self.srp.build(input_shape)
        self.gate.build(None)
        super().build(input_shape)
    def call(self, x, training=None):
        z_l = self.learn(x)
        z_f = self.freq(x) if self.enable_freq else tf.zeros_like(z_l)
        z_s = self.srp(self.mix(x)) if self.enable_spatial else tf.zeros_like(z_l)
        a = self.gate(z_f, z_s, z_l)
        #a = a + self.att2(a, training=True)   # attention after conv2
        a = a + self.att2(a, training=True)   # attention after conv2
        return a

# ---------- Contractive BasicBlock ----------
'''class HybridResBlock(tf.keras.layers.Layer):
    def __init__(self, filters, stride=1, econv_mode="pi", enable_freq=True, enable_spatial=True, **kw):
        super().__init__(**kw)
        self.f=int(filters); self.stride=int(stride)
        self.conv1 = CombinedConv(filters, stride=stride, econv_mode=econv_mode,
                                  enable_freq=enable_freq, enable_spatial=enable_spatial)
        self.act1  = GroupSort()
        self.conv2 = ECONV2D(filters, kernel_size=3, strides=1, use_bias=True, mode=econv_mode)
        self.act2  = GroupSort()
        self.proj  = None
        self.beta_logit = self.add_weight(name="beta_logit", shape=[], initializer=tf.zeros_initializer(), trainable=True)
    def build(self, input_shape):
        in_ch = int(input_shape[-1])
        if in_ch != self.f or self.stride != 1:
            self.proj = ECONV2D(self.f, kernel_size=1, strides=self.stride, use_bias=True, mode="pi")
            self.proj.build(input_shape)
        self.conv1.build(input_shape)
        self.conv2.build(self.conv1.compute_output_shape(input_shape))
        self.act1.build(self.conv1.compute_output_shape(input_shape))
        self.act2.build(self.conv2.compute_output_shape(self.act1.compute_output_shape(self.conv1.compute_output_shape(input_shape))))
        super().build(input_shape)
    def call(self, x, training=None):
        out = self.act1(self.conv1(x))
        out = self.act2(self.conv2(out))
        skip = self.proj(x) if self.proj is not None else x
        beta = tf.nn.sigmoid(self.beta_logit)
        return (1.0 - beta) * skip + beta * out'''

class HybridResBlock(tf.keras.layers.Layer):
    def __init__(self, filters, stride=1, econv_mode="pi",
                 enable_freq=True, enable_spatial=True, **kw):
        super().__init__(**kw)
        self.f = int(filters)
        self.stride = int(stride)
        self.conv1 = CombinedConv(filters, stride=stride, econv_mode=econv_mode,
                                  enable_freq=enable_freq, enable_spatial=enable_spatial)
        self.act1  = GroupSort()
        self.conv2 = ECONV2D(filters, kernel_size=3, strides=1, use_bias=True, mode=econv_mode)
        self.act2  = GroupSort()
        self.proj  = None
        self.beta_logit = self.add_weight(
            name="beta_logit", shape=[], initializer=tf.zeros_initializer(), trainable=True
        )
        self.att2  = TrainableCombinedAttentionLayer()

    def build(self, input_shape):
        in_ch = int(input_shape[-1])
        if in_ch != self.f or self.stride != 1:
            # projection only if shape changes
            self.proj = ECONV2D(self.f, kernel_size=1, strides=self.stride, use_bias=True, mode="pi")
        # no manual builds, no compute_output_shape calls
        super().build(input_shape)

    def call(self, x, training=None):
        out  = self.act1(self.conv1(x))
        out  = self.act2(self.conv2(out))
        skip = self.proj(x) if self.proj is not None else x
        beta = tf.nn.sigmoid(self.beta_logit)
        a = (1.0 - beta) * skip + beta * out
        a = a + self.att2(a, training=True)   # attention after conv2
        return a


# ---------- Combined ResNet-18 ----------
def build_combined_resnet18(input_shape=(224,224,3), num_classes=10,
                            widths=(64,128,256,512),
                            econv_mode_stages=("exact","pi","pi","pi"),
                            enable_freq_stages=(True, True, False, False),
                            enable_spatial_stages=(True, True, True, False)):
    inputs = tf.keras.Input(input_shape)
    x = CombinedConv(widths[0], stride=1, econv_mode=econv_mode_stages[0],
                     enable_freq=enable_freq_stages[0], enable_spatial=enable_spatial_stages[0],
                     name="stem")(inputs)
    x = GroupSort(name="stem_gs")(x)

    strides = [1,2,2,2]
    for i, (w, st) in enumerate(zip(widths, strides)):
        x = HybridResBlock(w, stride=st, econv_mode=econv_mode_stages[i],
                           enable_freq=enable_freq_stages[i],
                           enable_spatial=enable_spatial_stages[i],
                           name=f"stage{i}_b1")(x)
        #x  = TrainableCombinedAttentionLayer()(x)
        x = HybridResBlock(w, stride=1,  econv_mode=econv_mode_stages[i],
                           enable_freq=enable_freq_stages[i],
                           enable_spatial=enable_spatial_stages[i],
                           name=f"stage{i}_b2")(x)
        x  = TrainableCombinedAttentionLayer()(x)

    '''x = tf.keras.layers.AveragePooling2D(pool_size=4, name="avgpool")(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.expand_dims(tf.expand_dims(x, 1), 1)
    x = ECONV2D(num_classes, kernel_size=1, strides=1, use_bias=True, mode="pi", name="head")(x)
    logits = tf.squeeze(x, [1,2])
    return tf.keras.Model(inputs, logits, name="Combined_CRPF_ResNet18")'''

    #x  = TrainableCombinedAttentionLayer()(x)
    
    x = tf.keras.layers.Flatten()(x)
    #dense_layer = Dense(5, activation='selu')(flatten_layer)
    x = Dense(5, activation='softmax')(x)
    return tf.keras.Model(inputs, x, name="Combined_CRPF_ResNet18")

model = build_combined_resnet18(
    input_shape=(128,128,3), num_classes=5,
    econv_mode_stages=("pi","pi","pi","pi"),  # or ("freq_pi","freq_pi","pi","pi") if you added freq_pi
    enable_freq_stages=(True, True, False, False),
    enable_spatial_stages=(True, True, True, False)
)

model.summary()

In [ ]:
import tensorflow as tf

# ------------------------------------------------------------------
# 1️⃣  Hyper-parameters
# ------------------------------------------------------------------
sigma_train  = 0.5          # same σ you will use later for certification
batch_size   = 8
epochs       = 100


from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay

initial_gamma = 0.5
learning_rate = 1e-2
optimizer = Adam(learning_rate=0.001)
#opt = Adam(learning_rate=learning_rate, beta_1=0.9, beta_2=0.9, epsilon=None, amsgrad=False)
# Compile the model with the custom optimizer
model.compile(optimizer=optimizer,
              loss='categorical_crossentropy',
              #loss_weights=[initial_gamma, (1 -  initial_gamma)],
              metrics=['accuracy'], jit_compile=True)


from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
def checkpoint_callback():

    checkpoint_filepath = 'best1_model_cer_skin_lung.keras'

    model_checkpoint_callback= ModelCheckpoint(filepath=checkpoint_filepath,
                           save_weights_only=False,
                           #frequency='epoch',
                           monitor='val_loss',
                           save_best_only=True,
                            mode='min',
                           verbose=1)

    return model_checkpoint_callback

def early_stopping(patience):
    es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', restore_best_weights=True, patience=60, verbose=1)
    return es_callback



from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1,
                              patience=50, verbose = 1, min_lr=0.00001)

checkpoint_callback = checkpoint_callback()

early_stopping = early_stopping(patience=100)
callbacks = [checkpoint_callback, early_stopping, reduce_lr]
            


# ------------------------------------------------------------------
# 2️⃣  Build tf.data pipelines
# ------------------------------------------------------------------
gauss = tf.keras.layers.GaussianNoise(stddev=sigma_train)

def add_noise(x, y):
    """
    Add i.i.d. Gaussian noise **only during training**.
    Keras will pass 'training=True' when this dataset
    is used for .fit; no noise when you later call model(x, training=False).
    """
    x = gauss(x, training=True)        # <- noise goes here
    return x, y

gauss = tf.keras.layers.GaussianNoise(stddev=sigma_train)

def add_noise(x, y):
    x = tf.cast(x, tf.float32) / 255.0       # ①  uint8 → float32  [0,1]
    x = gauss(x, training=True)              # ②  additive N(0,σ²)
    return x, y

def preprocess_val(x, y):
    x = tf.cast(x, tf.float32) / 255.0       # same cast, but NO noise
    return x, y


train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train_s, y_train_s))
      .shuffle(len(X_train_s))
      .batch(batch_size)
      .map(add_noise,  num_parallel_calls=tf.data.AUTOTUNE)
      .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val_s, y_val_s))
      .batch(batch_size)
      .prefetch(tf.data.AUTOTUNE)      # **no** noise on validation set
)

# ------------------------------------------------------------------
# 3️⃣  Fit exactly as before
# ------------------------------------------------------------------
history = model.fit(
    train_ds,
    epochs         = epochs,
    validation_data= val_ds,
    verbose        = 1,
    callbacks      = callbacks,        # your ModelCheckpoint / EarlyStopping
    shuffle        = False             # shuffled already in the pipeline
)


In [ ]:
# hybrid_certifier.py
import tensorflow as tf, numpy as np
from scipy.stats import beta, norm

# ---------- One-sided lower Clopper–Pearson bound ----------
def lower_confidence_bound(k, n, alpha):
    # P(Bin(n, p) >= k) >= alpha  ⇒ lower bound on p
    return beta.ppf(alpha, k, n - k + 1) if k > 0 else 0.0

# ---------- Randomized smoothing ----------
class SmoothedClassifier:
    def __init__(self, base_model, sigma):
        """
        base_model must be DETERMINISTIC when called with training=False.
        sigma: std-dev of the Gaussian used BOTH for training & certification.
        """
        self.f = base_model
        self.sigma = float(sigma)
        self.num_classes = int(base_model.output_shape[-1])

    @tf.function(jit_compile=True)
    def _predict_noisy(self, x_batch):
        noise = tf.random.normal(tf.shape(x_batch)) * self.sigma
        logits = self.f(x_batch + noise, training=False)
        return tf.argmax(logits, axis=-1)

    def sample_class(self, x, N0=1000, batch=128):
        counts = np.zeros(self.num_classes, dtype=np.int32)
        reps = tf.repeat(x[None, ...], batch, axis=0)
        for done in range(0, N0, batch):
            m = min(batch, N0 - done)
            preds = self._predict_noisy(reps[:m])
            counts += np.bincount(preds.numpy(), minlength=self.num_classes)
        return int(np.argmax(counts))

    def certify(self, x, N=100_000, N0=1000, alpha=1e-3, batch=512):
        A = self.sample_class(x, N0=N0, batch=batch)
        countA = 0
        reps = tf.repeat(x[None, ...], batch, axis=0)
        for done in range(0, N, batch):
            m = min(batch, N - done)
            preds = self._predict_noisy(reps[:m])
            countA += int(np.sum(preds.numpy() == A))
        pA_low = lower_confidence_bound(countA, N, alpha)
        if pA_low <= 0.5:
            return 0.0
        return self.sigma * norm.ppf(pA_low)

# ---------- Deterministic (Lipschitz) branch ----------
def deterministic_radius(model, x, L=1.0):
    """
    r_det = margin / (2L), where margin = top1 - top2 logit.
    Only valid if your network has a PROVABLE Lipschitz constant ≤ L.
    """
    logits = model(x[None, ...], training=False)  # deterministic forward
    top2 = tf.nn.top_k(logits, k=2).values
    margin = float((top2[:, 0] - top2[:, 1])[0].numpy())
    return max(margin / (2.0 * float(L)), 0.0)

# ---------- Hybrid wrapper ----------
class HybridCertifier:
    def __init__(self, base_model, sigma, L=1.0):
        self.smooth = SmoothedClassifier(base_model, sigma)
        self.model  = base_model
        self.L      = float(L)

    @staticmethod
    def assert_deterministic(model, x, atol=1e-6):
        """Catches accidental randomness when training=False."""
        y1 = model(x[None, ...], training=False)
        y2 = model(x[None, ...], training=False)
        tf.debugging.assert_near(y1, y2, atol=atol,
            message="Model appears stochastic at eval; gate all noise with `if training:`.")

    def certify(self, x, N=100_000, N0=1000, alpha=1e-3, batch=512):
        # Optional health check (run once per script, not per sample)
        # self.assert_deterministic(self.model, x)

        r_det    = deterministic_radius(self.model, x, L=self.L)
        r_smooth = self.smooth.certify(x, N=N, N0=N0, alpha=alpha, batch=batch)
        return max(r_det, r_smooth), r_det, r_smooth


In [ ]:
# Build a clean test dataset like your val pipeline
def preprocess(x, y):
    #return tf.cast(x, tf.float32) / 255.0, y
    return x, y #tf.cast(x, tf.float32) / 255.0, y

test_ds = (tf.data.Dataset.from_tensor_slices((X_test_s, y_test_s))
             .batch(1)
             .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
             .prefetch(tf.data.AUTOTUNE))

# HYBRID certification
from hybrid_certifier import HybridCertifier
SIGMA = 0.5     # MUST equal the sigma you used in GaussianNoise during training
L     = 1.0     # Set to 1.0 only if every layer is provably ≤1-Lipschitz

hybrid = HybridCertifier(model, sigma=SIGMA, L=L)

clean_ok = det_ok = smooth_ok = hybrid_ok = 0
N0, N, alpha = 1_000, 100_000, 1e-3

for x, y in test_ds:
    true = int(tf.argmax(y, -1).numpy()[0])
    pred = int(tf.argmax(model(x, training=False), -1).numpy()[0])
    clean_ok += int(pred == true)

    r_best, r_det, r_smooth = hybrid.certify(tf.squeeze(x, 0), N=N, N0=N0, alpha=alpha, batch=256)
    det_ok    += int((r_det    > 0) and (pred == true))
    smooth_ok += int((r_smooth > 0) and (pred == true))
    hybrid_ok += int((r_best   > 0) and (pred == true))

n_test = len(X_test_s)
print(f"\nClean acc              : {clean_ok/n_test:.3%}")
print(f"Deterministic cert acc : {det_ok  /n_test:.3%}")
print(f"Smoothing    cert acc  : {smooth_ok/n_test:.3%}")
print(f"HYBRID       cert acc  : {hybrid_ok/n_test:.3%}  (α={alpha}, σ={SIGMA})")


In [ ]:
# hybrid_certify.py
import tensorflow as tf, numpy as np, math
from pathlib import Path
from scipy.stats import beta, norm    # pip install scipy

# ----------------------------------------------------------------------
# 0.  Hyper-parameters
# ----------------------------------------------------------------------
SIGMA      = 0.5          # std-dev for input Gaussian noise  (train + certify)
LIPSCHITZ  = 1.0          # global Lipschitz constant of your network
EPOCHS     = 50
BATCH_TRAIN= 128
N0, N      = 1_000, 100_000   # Monte-Carlo sample counts (CIFAR-10 defaults)
ALPHA      = 1e-3             # failure prob → 99.9 % confidence
CKPT_FILE  = "hybrid_noise.keras"

# ----------------------------------------------------------------------
# 1.  Data  (CIFAR-10 for demo; swap in your own dataset)
# ----------------------------------------------------------------------

#(x_tr, y_tr), (x_te, y_te) = tf.keras.datasets.cifar10.load_data()
#C = 10
#x_tr = x_tr.astype("float32")/255.0; x_te = x_te.astype("float32")/255.0
#y_tr = tf.keras.utils.to_categorical(y_tr, C); y_te = tf.keras.utils.to_categorical(y_te, C)
#train_ds = tf.data.Dataset.from_tensor_slices((x_tr, y_tr)).shuffle(50_000).batch(BATCH_TRAIN)
#test_ds  = tf.data.Dataset.from_tensor_slices((x_te, y_te)).batch(1)  # certify one-by-one

# ----------------------------------------------------------------------
# 2.  Model  (⇢ put **your own architecture** here)
# ----------------------------------------------------------------------
#model      = make_model()
loss_fn    = tf.keras.losses.CategoricalCrossentropy(from_logits=True)
optimizer  = tf.keras.optimizers.Adam(1e-3)
noise_layer= tf.keras.layers.GaussianNoise(SIGMA)      # input noise

# ----------------------------------------------------------------------
# 3.  Training loop (Gaussian-noise augmentation every batch)
# ----------------------------------------------------------------------
@tf.function(jit_compile=True)
def train_step(x, y):
    with tf.GradientTape() as tape:
        x_noisy = noise_layer(x, training=True)
        logits  = model(x_noisy, training=True)
        loss    = loss_fn(y, logits)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    acc = tf.reduce_mean(tf.cast(tf.argmax(logits,-1)==tf.argmax(y,-1), tf.float32))
    return loss, acc

for epoch in range(1, EPOCHS+1):
    tot_l, tot_a, n = 0., 0., 0
    for xb, yb in train_ds:
        l,a = train_step(xb, yb)
        bs  = xb.shape[0]
        tot_l += l.numpy()*bs; tot_a += a.numpy()*bs; n += bs
    print(f"Epoch {epoch:02d}: loss={tot_l/n:.4f}  noise-train acc={tot_a/n:.3%}")

model.save(CKPT_FILE)
print("✓ trained weights saved to", CKPT_FILE)

# ----------------------------------------------------------------------
# 4.  Certification utilities
# ----------------------------------------------------------------------
def clopper_low(k, n, alpha): return beta.ppf(alpha/2, k, n-k+1) if k>0 else 0.
def clopper_up (k, n, alpha): return beta.ppf(1-alpha/2, k+1, n-k) if k<n else 1.

@tf.function(jit_compile=True)
def predict_noisy_batch(x_batch, sigma):
    noise  = tf.random.normal(tf.shape(x_batch))*sigma
    logits = model(x_batch+noise, training=False)
    return tf.argmax(logits, axis=-1)          # int32

def smoothed_radius(x, sigma=SIGMA, N=N, N0=N0, alpha=ALPHA, batch=512):
    # Step-1: majority class
    counts = np.zeros(C, dtype=np.int32)
    reps   = tf.repeat(x[None,...], batch, axis=0)
    for done in range(0, N0, batch):
        m = min(batch, N0-done)
        preds = predict_noisy_batch(reps[:m], sigma)
        counts += np.bincount(preds.numpy(), minlength=C)
    A = counts.argmax()

    # Step-2: estimate p_A
    k_A = 0
    for done in range(0, N, batch):
        m = min(batch, N-done)
        preds = predict_noisy_batch(reps[:m], sigma)
        k_A += int(np.sum(preds.numpy()==A))
    pA_low = clopper_low(k_A, N, alpha)
    pB_up  = clopper_up (N-k_A, N, alpha)
    if pA_low<=pB_up:                   # certificate fails
        return 0.0
    return sigma * norm.ppf(pA_low)

def deterministic_radius(x, L=LIPSCHITZ):
    logits = model(x[None,...], training=False)
    top2   = tf.nn.top_k(logits, k=2).values
    margin = (top2[:,0]-top2[:,1])[0].numpy()   # scalar
    return margin/(2.0*L)

def hybrid_radius(x):
    r_det   = deterministic_radius(x)
    r_smooth= smoothed_radius(x)
    return max(r_det, r_smooth), r_det, r_smooth

# ----------------------------------------------------------------------
# 5.  Evaluate clean, deterministic, smoothing, and hybrid accuracies
# ----------------------------------------------------------------------
clean_ok, det_ok, smooth_ok, hybrid_ok = 0,0,0,0

for (x,y) in test_ds:
    true  = tf.argmax(y, -1).numpy()[0]
    pred  = tf.argmax(model(x, training=False), -1).numpy()[0]
    clean_ok += int(pred==true)

    r_h, r_d, r_s = hybrid_radius(tf.squeeze(x,0))
    det_ok    += int((r_d>0)    and (pred==true))
    smooth_ok += int((r_s>0)    and (pred==true))
    hybrid_ok += int((r_h>0)    and (pred==true))

N_TEST = len(x_te)
print("\n=== RESULTS (α = {:.0e}, σ = {:.2f}) ===".format(ALPHA,SIGMA))
print(f"Clean accuracy        : {clean_ok / N_TEST:.3%}")
print(f"Deterministic cert -r : {det_ok   / N_TEST:.3%}")
print(f"Smoothing    cert -r : {smooth_ok/ N_TEST:.3%}")
print(f"HYBRID       cert -r : {hybrid_ok/ N_TEST:.3%}")


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent

# 1) Make sure inputs are float32 in [0,1]
def to_float01(x):
    x = tf.convert_to_tensor(x)
    if x.dtype.is_integer:
        x = tf.cast(x, tf.float32) / 255.0
    else:
        x = tf.cast(x, tf.float32)
    return x

X_test_f = to_float01(X_test_s)
y_test_f = y_test_s
# 2) Labels as int (not one-hot) for metrics; keep one-hot separately if you want AUC-OVR
'''if 'y_test_one_hot' in globals():
    y_test_labels = np.argmax(y_test_one_hot, axis=1) if y_test_one_hot.ndim > 1 else np.asarray(y_test_one_hot).astype(int)
else:
    y_test_labels = np.asarray(y_test_s).astype(int)
'''
batch_size = 64
num_samples = len(X_test_f)
num_batches = (num_samples + batch_size - 1) // batch_size

epsilon_values = [0.016, 0.0313, 0.047, 0.0627]  # feel free to add more
results_per_epsilon = []

for epsilon in epsilon_values:
    adv_batches = []

    for i in range(num_batches):
        s = i * batch_size
        e = min((i + 1) * batch_size, num_samples)

        x_b = X_test_f[s:e]
        y_b = np.argmax(y_test_f[s:e], axis=1).astype("int32") #y_test_labels[s:e]

        # CleverHans expects tf.Tensors, float32, 0..1; set clip_min/max
        adv_b = projected_gradient_descent(
            model_fn=model,
            x=x_b,
            eps=tf.cast(epsilon, tf.float32),
            eps_iter=tf.cast(epsilon/4.0, tf.float32),
            nb_iter=10,                        # use >1 step for PGD (e.g., 40)
            norm=np.inf,
            loss_fn=None,
            clip_min=0.0,
            clip_max=1.0,
            y=tf.convert_to_tensor(y_b, dtype=tf.int32),
            targeted=False,
            rand_init=True,                    # typical PGD with random start
            rand_minmax=epsilon,
            sanity_checks=False,
        )
        adv_batches.append(adv_b.numpy())

    X_adv = np.concatenate(adv_batches, axis=0)

    # 3) Predict on adversarial set
    logits = model.predict(X_adv, batch_size=128, verbose=0)

    # 4) Handle binary vs multiclass automatically
    y_pred = tf.squeeze(logits)
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_f) * 100
    precision = precision_score(y_pred_binary, y_test_f, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_f, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_f, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)

In [ ]:
y_pred = tf.squeeze(logits)
y_pred_binary = y_pred >= 0.5
y_pred_binary = np.array(y_pred_binary, dtype='int32')

# Calculate evaluation metrics for the current epsilon
accuracy = accuracy_score(y_pred_binary, y_test_f) * 100
precision = precision_score(y_pred_binary, y_test_f, average='macro') * 100
recall = recall_score(y_pred_binary, y_test_f, average='macro') * 100
f1 = f1_score(y_pred_binary, y_test_f, average='macro') * 100
#auc = roc_auc_score(y_pred, y_test_one_hot) * 100

# Store results for the current epsilon
results_per_epsilon.append({
    'epsilon': epsilon,
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1': f1,
    #'auc': auc
})

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
y_pred = tf.squeeze(logits)
y_pred_binary = y_pred >= 0.5
y_pred_binary = np.array(y_pred_binary, dtype='int32')

# Calculate evaluation metrics for the current epsilon
accuracy = accuracy_score(y_pred_binary, y_test_f) * 100
precision = precision_score(y_pred_binary, y_test_f, average='macro') * 100
recall = recall_score(y_pred_binary, y_test_f, average='macro') * 100
f1 = f1_score(y_pred_binary, y_test_f, average='macro') * 100
#auc = roc_auc_score(y_pred, y_test_one_hot) * 100

# Store results for the current epsilon
results_per_epsilon.append({
    'epsilon': epsilon,
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1': f1,
    #'auc': auc
})

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)
